1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split 
from tensorflow.keras import Input, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

2. Load Data

In [2]:
# dataset path
DATA_PATH = "../../annotations"
os.listdir(DATA_PATH)

['lisa_dataset_split.csv', 'new_annotations.csv']

In [3]:
# Load the precomputed split.
# Expects 'lisa_dataset_split.csv' to be located under DATA_PATH.
df = pd.read_csv(os.path.join(DATA_PATH,'new_annotations.csv'), sep=';')

In [4]:
# Preview first 5 rows
df.head()

,image_id,label,isNight,split
0,../../datasets/processed_images\train\img_0.jpg,1,0,train
1,../../datasets/processed_images\train\img_1.jpg,1,0,train
2,../../datasets/processed_images\train\img_2.jpg,1,0,train
3,../../datasets/processed_images\train\img_3.jpg,1,0,train
4,../../datasets/processed_images\train\img_4.jpg,1,0,train


In [5]:
# Shape: (rows, cols)
df.shape

(51826, 4)

3. Build CNN-ready datasets

In [6]:
CSV_PATH   = "../../annotations/new_annotations.csv"
IMG_SIZE   = 50
NUM_CLASSES = 3  # stop / warning / go mapped to 0..2

# Auto-detect separator (handles ';' from previous exports)
df = pd.read_csv(CSV_PATH, sep=None, engine="python")

# Quick sanity checks on splits and label balance
print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

# Labels come as 1/2/3 -> remap to 0/1/2
df["label"] = df["label"].astype(int) - 1
assert df["label"].between(0, NUM_CLASSES-1).all(), "Found labels outside 0..2"

def build_Xy(df_part):
    """
    Build design matrix and one-hot targets from a dataframe slice.

    - Reads each image path, resizes to IMG_SIZE x IMG_SIZE, normalizes to [0,1]
    - Flattens the image to a single vector
    - Appends the binary context feature `isNight` as an extra column
    - One-hot encodes labels to shape (N, NUM_CLASSES)
    """
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label"])
        is_night = int(row["isNight"])

        # Skip missing or unreadable images
        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        # Resize and normalize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0

        # Collect features and target
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    # Flatten images: (N, H, W, C) -> (N, H*W*C)
    X_images = np.array(X_images).reshape(len(X_images), -1)
    # Context feature as (N, 1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    # Concatenate image vector with isNight -> (N, H*W*C + 1)
    X_combined = np.hstack([X_images, X_isNight])

    # One-hot encode targets
    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y

# Build split-specific datasets from the 'split' column
X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "val"])
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)


Splits: {'train': 30707, 'test': 14433, 'val': 6686}
Labels per split:
 label      1    2      3
split                   
test    7229  499   6705
train  14272  800  15635
val     2681  256   3749


100%|██████████| 14433/14433 [00:03<00:00, 4450.57it/s]


Shapes: 
 X_train: (30707, 7501)  y_train: (30707, 3) 
 X_val:   (6686, 7501)  y_val:   (6686, 3) 
 X_test:  (14433, 7501)  y_test:  (14433, 3)


4. Baseline CNN classifier

In [7]:
IMG_SIZE = 50
PIXELS = IMG_SIZE * IMG_SIZE * 3  # 50x50 RGB → 7,500 features

def split_back(X):
    """
    Undo the earlier feature concatenation:
    - First PIXELS columns: flattened image (HxWxC).
    - Last column: auxiliary scalar feature (isNight ∈ {0,1}).

    Returns
    -------
    X_img : np.ndarray, shape (N, IMG_SIZE, IMG_SIZE, 3)
        Image tensor reconstructed from the flat vector.
    X_isn : np.ndarray, shape (N, 1)
        Auxiliary feature as a column vector (isNight).
    """
    # Slice out the flattened image part and reshape to HxWxC
    X_img_flat = X[:, :PIXELS]
    X_img = X_img_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 3).astype("float32")

    # Take the last column as the isNight flag and keep it as (N,1)
    X_isn = X[:, -1].reshape(-1, 1).astype("float32")   
    return X_img, X_isn

# Apply to each split
Ximg_train, isn_train = split_back(X_train)
Ximg_val,   isn_val   = split_back(X_val)
Ximg_test,  isn_test  = split_back(X_test)

# Sanity check: expect (N, 50, 50, 3) for images and (N, 1) for isNight
print(Ximg_train.shape, isn_train.shape)  

(30707, 50, 50, 3) (30707, 1)


CNN with auxiliary metadata input (isNight) for multi-input classification

In [8]:
# Number of target classes inferred from one-hot labels
NUM_CLASSES = y_train.shape[1]

# ----- Image branch ----
img_in = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = Conv2D(32, 3, activation='relu')(img_in)    # low-level features
x = MaxPooling2D()(x)                           # downsample spatial dims
x = BatchNormalization()(x)                     # stabilize / speed up training

x = Conv2D(64, 3, activation='relu')(x)         # mid-level features
x = MaxPooling2D()(x)                           
x = BatchNormalization()(x)    

x = Conv2D(128, 3, activation='relu')(x)        # high-level features
x = MaxPooling2D()(x)                              
x = Flatten()(x)                                # to vector for fusion

# ----- Auxiliary metadata branch -----
isn_in = Input(shape=(1,), name="isNight")      # scalar context feature ∈ {0,1}

# ----- Fusion + classifier head -----
h = Concatenate()([x, isn_in])                  # fuse visual + context features
h = Dense(128, activation='relu')(h)            # joint representation
h = Dropout(0.3)(h)                             # regularization
out = Dense(NUM_CLASSES, activation='softmax')(h)

# Build & compile multi-input model
cnn_model = Model([img_in, isn_in], out)
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()




Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image (InputLayer)          [(None, 50, 50, 3)]          0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 48, 48, 32)           896       ['image[0][0]']               
                                                                                                  
 max_pooling2d (MaxPooling2  (None, 24, 24, 32)           0         ['conv2d[0][0]']              
 D)                                                                                               
                                                                                                  
 batch_normalization (Batch  (None, 24, 24, 32)           128       ['max_pooling2d[0][0]']

5. Training loop with early stopping

In [9]:
# Stop training when validation loss hasn’t improved for 5 epochs,
# and roll back to the best weights observed during training.
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# ----- Fit -----
history_cnn = cnn_model.fit(
    [Ximg_train, isn_train], y_train,           # multi-input: images + isNight
    validation_data=([Ximg_val, isn_val], y_val),
    epochs=50, 
    batch_size=64, 
    shuffle=True,                               # shuffle each epoch
    callbacks=[early_stop], 
    verbose=1
)

Epoch 1/50


480/480 [==============================] - 19s 37ms/step - loss: 0.0233 - accuracy: 0.9936 - val_loss: 0.0113 - val_accuracy: 0.9988
Epoch 2/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0081 - accuracy: 0.9981 - val_loss: 0.0109 - val_accuracy: 0.9991
Epoch 3/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0055 - accuracy: 0.9986 - val_loss: 0.0195 - val_accuracy: 0.9987
Epoch 4/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0052 - accuracy: 0.9986 - val_loss: 0.0933 - val_accuracy: 0.9856
Epoch 5/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0024 - accuracy: 0.9996 - val_loss: 0.0108 - val_accuracy: 0.9994
Epoch 6/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0025 - accuracy: 0.9994 - val_loss: 0.0174 - val_accuracy: 0.9993
Epoch 7/50
480/480 [==============================] - 17s 36ms/step - loss: 0.0058 - accuracy: 0.9992 - val_loss: 1.2575 - val_accurac

6. Evaluation

In [10]:
# Evaluate on held-out test set
test_loss_cnn, test_acc_cnn = cnn_model.evaluate([Ximg_test, isn_test], y_test, verbose=0)
print(f"CNN → Test loss: {test_loss_cnn:.4f} | Test acc: {test_acc_cnn:.4f}")

CNN → Test loss: 0.0068 | Test acc: 0.9997


In [11]:

# Predict class probabilities on the test set (batched for speed/memory).
y_pred_cnn = cnn_model.predict([Ximg_test, isn_test], batch_size=512, verbose=0)

# Convert one-hot (or probas) to hard class indices.
y_pred_idx = y_pred_cnn.argmax(axis=1)
y_true_idx = y_test.argmax(axis=1)

# Per-class precision/recall/F1 + macro/micro averages.
print("Classification report (CNN):")
print(classification_report(y_true_idx, y_pred_idx, digits=4))

# Raw confusion matrix counts: rows = true, columns = predicted.
print("Confusion matrix (CNN):")
print(confusion_matrix(y_true_idx, y_pred_idx))

Classification report (CNN):
              precision    recall  f1-score   support

           0     0.9994    0.9999    0.9997      7229
           1     0.9980    0.9980    0.9980       499
           2     1.0000    0.9996    0.9998      6705

    accuracy                         0.9997     14433
   macro avg     0.9991    0.9991    0.9991     14433
weighted avg     0.9997    0.9997    0.9997     14433

Confusion matrix (CNN):
[[7228    1    0]
 [   1  498    0]
 [   3    0 6702]]


7. Save the trained model

In [14]:
import os

MODEL_DIR = "../../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "traffic_light_cnn.keras")

cnn_model.save(MODEL_PATH)